## Linking CISA scenarios with MITRE ATT&CK (Goal 2)

This notebook connects each CISA scenario event to MITRE ATT&CK techniques, and through them to MITRE mitigations (possible responses). Together with the business impacts from `5_goal1_scenario_analysis`, this gives the structured chain:

```
Exercise ──< Event ──< Technique ──< Mitigation
             │  (threat type)      (possible response)
             └── Business impact
                 (described consequence)
```

**What this model does and does not show:** it links an event to *possible* responses (MITRE mitigations for the techniques behind it) and, separately, to the consequence the scenario *describes*. It does **not** show that choosing response A leads to outcome B. The CISA manuals contain no evidence about how well a response works, so that cannot be estimated from this data.

**Main design choices:**
- **TF-IDF is kept only as a baseline.** Its results are inspected in steps 2–3 and turn out to be unreliable: the CISA manuals describe events in business language ("computers display a red screen"), while MITRE uses technical language ("adversaries may encrypt data"), so few words overlap.
- **The actual link is a team-created mapping** from each CISA `threat_type` to MITRE techniques, with a short justification for every choice (step 4). Each link also has a **`mapping_status`**: *supported* when the scenario text clearly describes the technique, *plausible* when it fits the threat type but is not stated in the events.
- **`threat_type` is used instead of `threat_category`.** Some categories are too broad: e.g. *Other Security Incident* contains a data-integrity issue, suspicious network traffic and a stolen laptop, which correspond to completely different techniques.
- **ICS ATT&CK is used for the Industrial Control Systems manual**, in addition to Enterprise ATT&CK (attacks on industrial systems usually start in the IT network).
- **The reviewed threat and impact labels** from `5_goal1_scenario_analysis` are used, so Goals 1, 2 and 3 all use the same labels.

Run order: `2_mitreFileReading_DONE` (MITRE files with the `domain` column) and `5_goal1_scenario_analysis` (reviewed labels) must have been run first.

#### 1. Loading CISA and MITRE data
The classified CISA events, the reviewed labels and the MITRE techniques and mitigations are loaded.

*Why:* the reviewed threat and impact labels from `5_goal1_scenario_analysis` replace the original ones from `3_GivingEventsThreatLevel-CISA`, so all three goals use the same labels. The `impact_status` column (described as occurring / potential or unconfirmed / none reported) is kept, so outcomes can be read correctly later. The check stops with a clear message if the MITRE files are still from the old Enterprise-only version.

In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data/processed")
if not data_dir.exists():
    data_dir = Path("data/processed")

original = pd.read_csv(data_dir / "cisa_classified_events.csv")
techniques = pd.read_csv(data_dir / "mitre_techniques.csv")
technique_mitigations = pd.read_csv(data_dir / "technique_mitigations.csv")

assert "domain" in techniques.columns, "Run 2_mitreFileReading_DONE first (Enterprise + ICS)."
assert (data_dir / "cisa_events_reviewed.csv").exists(), "Run 5_goal1_scenario_analysis first (reviewed labels)."

# use the reviewed labels from the Goal 1 notebook, so all three goals use the same labels
reviewed = pd.read_csv(data_dir / "cisa_events_reviewed.csv")
events = original.merge(reviewed, on="inject_id", validate="one_to_one")
events["threat_category"] = events["reviewed_threat"]
events["impact_category"] = events["reviewed_impact"]

print("CISA events:", len(events))
print("Threat changed in the review:", (events["threat_category"] != original["threat_category"]).sum(),
      "| impact changed in the review:", (events["impact_category"] != original["impact_category"]).sum())
print("MITRE techniques per domain:")
print(techniques["domain"].value_counts().to_string())

pd.set_option("display.max_colwidth", 120)

CISA events: 66
Threat changed in the review: 1 | impact changed in the review: 4
MITRE techniques per domain:
domain
enterprise-attack    706
ics-attack           104


#### 2. Baseline: comparing events and techniques with TF-IDF
TF-IDF and cosine similarity are used to find the MITRE technique whose description is most similar to each CISA event.

*Why:* this is the original approach, kept as a baseline so it can be compared with the team-created mapping later (step 8). Only Enterprise techniques are used here, as in the original version.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ent = techniques[techniques["domain"] == "enterprise-attack"].reset_index(drop=True)

event_texts = events["event_description"].fillna("")
technique_texts = ent["name"].fillna("") + " " + ent["description"].fillna("")

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
tfidf = vectorizer.fit_transform(pd.concat([event_texts, technique_texts], ignore_index=True))

event_vectors = tfidf[:len(events)]
technique_vectors = tfidf[len(events):]
similarities = cosine_similarity(event_vectors, technique_vectors)

print("Similarity matrix:", similarities.shape)

Similarity matrix: (66, 706)


#### 3. Inspecting the TF-IDF matches
The best-matching technique and its similarity score are shown for each event.

*Why:* before using the results, it is necessary to check whether they make sense. Mostly they do not: a few matches are right (e.g. *Data Encrypted for Impact* for some ransomware events), but many are not (e.g. *Weaken Encryption* for a ransomware event, *Determine Physical Locations* for a suspected ransomware event), and the similarity scores are very low (median around 0.05, where 1.0 would be identical text). This is the reason a team-created mapping is used instead.

In [3]:
best = similarities.argmax(axis=1)
tfidf_top = events[["inject_id", "threat_type"]].copy()
tfidf_top["tfidf_technique"] = ent.loc[best, "technique_id"].values + " " + ent.loc[best, "name"].values
tfidf_top["similarity"] = similarities.max(axis=1).round(3)

print("Highest similarity score:", tfidf_top["similarity"].max())
print("Median similarity score:", tfidf_top["similarity"].median())
display(tfidf_top[tfidf_top["threat_type"].str.contains("Ransomware|Phishing|DDoS")])

Highest similarity score: 0.18
Median similarity score: 0.0515


,inject_id,threat_type,tfidf_technique,similarity
1,HC-INJ-89,Suspected Phishing,T1204.001 Malicious Link,0.068
8,HC-INJ-156,Ransomware,T1486 Data Encrypted for Impact,0.065
22,RW-INJ-90,Suspected Phishing,T1586.002 Email Accounts,0.055
24,RW-INJ-124,Ransomware,T1600 Weaken Encryption,0.048
26,RW-INJ-128,Ransomware,T1486 Data Encrypted for Impact,0.029
39,EX-INJ-109,Whaling / Ransomware,T1589.003 Employee Names,0.090
43,ICS-INJ-116,Suspected Phishing,T1586.002 Email Accounts,0.095
47,ICS-INJ-146,Suspected Ransomware,T1591.001 Determine Physical Locations,0.041
51,FS-INJ-98,Suspected Phishing,T1586.002 Email Accounts,0.086
54,FS-INJ-123,Ransomware,T1534 Internal Spearphishing,0.041


#### 4. Team-created mapping: threat types → MITRE techniques
Each CISA threat type is mapped to the MITRE techniques that describe it, with a short justification.

*Why:* this replaces the TF-IDF matching as the actual link between the datasets. Each choice is based on reading both the CISA events of that type and the MITRE technique descriptions. It was made by the team, not by a security expert, so it is called a *team-created* mapping.

**Mapping status.** Every link gets a `mapping_status`:
- **supported:** the scenario text clearly describes the technique (e.g. *"control data is encrypted"* → T1486 Data Encrypted for Impact for the ICS ransomware event).
- **plausible:** the technique fits the threat type, but the events do not state it. For example, a ransom demand alone does not show that backups were disabled (T1490 Inhibit System Recovery), and a data breach does not always come from cloud storage (T1530). These links are kept as candidates, but should not be presented as confirmed.

The status is checked **per event**: `supported_evidence` lists, for each supported event–technique link, the words in that event that describe the technique. The same technique is *plausible* in other events of that threat type whose text does not describe it (e.g. T1068 is supported in IT-INJ-129, where the exploited vulnerability is described, but only plausible in IT-INJ-127, where the account is just discovered). In the mapping table below, a link is *supported* if at least one event has evidence for it.

Some threat types are intentionally left **without** techniques:
- *Security Alert*, *No Specific Threat*: no attack takes place in these events.
- *Accidental Data Disclosure*, *Device Theft*: ATT&CK only describes deliberate actions by attackers in a network, so an employee mistake or a stolen laptop has no matching technique.

The ICS mapping is applied **in addition** to the Enterprise mapping for the Industrial Control Systems manual.

In [4]:
# threat_type: ([technique IDs], justification)
enterprise_map = {
    "Suspected Phishing": (["T1566", "T1598"],
        "Emails trick employees into opening attachments or entering credentials."),
    "Whaling / Ransomware": (["T1566", "T1486"],
        "Phishing aimed at executives that leads to ransomware."),
    "Ransomware": (["T1486", "T1490"],
        "Files are encrypted and a ransom is demanded; recovery is often blocked."),
    "Suspected Ransomware": (["T1486", "T1490"],
        "Same behaviour as ransomware, not yet confirmed."),
    "Extortion Threat": (["T1657"],
        "Attackers demand payment; MITRE covers extortion under Financial Theft."),
    "Insider Threat": (["T1078", "T1098"],
        "A trusted employee misuses legitimate accounts and access rights."),
    "Potential Insider Threat": (["T1078"],
        "A terminated or unhappy employee may still have valid access."),
    "Unauthorized Access": (["T1136", "T1068"],
        "An unknown administrator account is created by exploiting a vulnerability."),
    "Unauthorized Access Risk": (["T1078", "T1621"],
        "Old accounts left active (valid accounts) and repeated MFA push requests."),
    "Unauthorized File Deletion": (["T1485"],
        "Files crucial to core functions are deleted."),
    "Unauthorized File Modification": (["T1565.001", "T1485"],
        "Files on shared drives are renamed, moved or deleted."),
    "Data Breach": (["T1530", "T1567"],
        "Customer or patient data is taken from cloud/network storage and leaked."),
    "Suspected Data Exfiltration": (["T1048", "T1071.004"],
        "Unusual DNS traffic used to move data out of the network."),
    "Accidental Data Disclosure": ([],
        "Employee mistake, not an attacker technique."),
    "DDoS": (["T1498", "T1499", "T1584.005"],
        "Millions of requests overload servers; the attack uses a botnet."),
    "DDoS / Unintentional TDoS": (["T1498", "T1499.002"],
        "Network flooding that also overloads the call centre."),
    "Data Integrity Issue": (["T1565.001"],
        "Patient records show incorrect information (stored data manipulation)."),
    "Device Theft": ([],
        "Physical theft of a laptop, outside the scope of ATT&CK."),
    "Suspicious Network Activity": (["T1041"],
        "Months of anomalous traffic, consistent with data leaving via a C2 channel."),
    "Medical Device Malfunction": (["T1565"],
        "Monitors show inaccurate data and pumps misbehave (data manipulation)."),
    "System Disruption (Unconfirmed Cause)": (["T1489", "T1499"],
        "Services stop or slow down; the cause is not yet known."),
    "Supply Chain Compromise": (["T1195.002", "T1199"],
        "Attackers reach customers through a managed service provider's software."),
    "Suspicious Third-Party Activity": (["T1199", "T1078"],
        "A vendor's address or access is used for unauthorized activity."),
    "Third-Party Data Breach": (["T1199"],
        "Data is exposed through partners the organization trusts."),
    "Vulnerability Exposure": (["T1190", "T1068"],
        "Unpatched or unsupported software/hardware that attackers can exploit."),
    "Security Alert": ([], "Warning from authorities, no attack yet."),
    "No Specific Threat": ([], "No attack in this event."),
    "No Specific Cyber Threat": ([], "No attack in this event."),
}

# extra ICS techniques, only for the Industrial Control Systems manual
ics_map = {
    "Suspected Phishing": (["T0865"], "Phishing aimed at IT and OT staff."),
    "Unauthorized Access Risk": (["T0859"], "Approved MFA requests give access to valid accounts in the OT environment."),
    "Suspicious Third-Party Activity": (["T0862"], "Equipment from an OT vendor may be compromised."),
    "System Disruption (Unconfirmed Cause)": (["T0826", "T0832"], "Service disruptions and distorted data inputs (manipulation of view)."),
    "Suspected Ransomware": (["T0816", "T0826", "T0828"], "Fail-safes shut down operations, causing lost availability and revenue."),
    "Extortion Threat": (["T0831", "T0828"], "Attackers take credit for malfunctioning industrial processes and threaten to continue."),
    "Data Breach": (["T0883", "T0822"], "Vulnerable internet-facing remote access solution."),
}

def to_table(mapping, domain):
    rows = [(tt, tid, domain, why) for tt, (ids, why) in mapping.items() for tid in (ids or [None])]
    return pd.DataFrame(rows, columns=["threat_type", "technique_id", "domain", "justification"])

threat_technique_map = pd.concat([to_table(enterprise_map, "enterprise-attack"),
                                  to_table(ics_map, "ics-attack")], ignore_index=True)

# Evidence for "supported" links, checked event by event: (inject_id, technique_id) -> words from the event text.
# A link is only "supported" in the event whose text describes it; in all other events it is "plausible".
supported_evidence = {
    ("HC-INJ-89", "T1566"): "The e-mail contains a link and attachment",
    ("RW-INJ-90", "T1566"): "an email ... instructs them to access a PDF",
    ("FS-INJ-98", "T1566"): "another email with a PDF attachment",
    ("CM-INJ-95", "T1566"): "employees click on the embedded link",
    ("ICS-INJ-116", "T1566"): "email that appears to be from the CEO ... log in to a website",
    ("EX-INJ-109", "T1566"): "what appears to be an email from another senior-level employee",
    ("VD-INJ-111", "T1199"): "ransomware attacks against MSPs and their downstream customers",
    ("VD-INJ-162", "T1195.002"): "customers unknowingly had malware ... from the MSP's platform",
    ("VD-INJ-162", "T1199"): "any user of the MSP's platform could be affected",
    ("RW-INJ-122", "T1048"): "DNS traffic ... data was sent ... to external IP addresses",
    ("IT-INJ-127", "T1136"): "presence of an unauthorized administrator-level account",
    ("IT-INJ-129", "T1136"): "create a new administrator account",
    ("IT-INJ-129", "T1068"): "added to the system by exploiting the microprocessor vulnerability",
    ("EX-INJ-126", "T1078"): "an administrator account ... belonged to the terminated employee",
    ("ICS-INJ-146", "T1486"): "<control logic, historian> data is encrypted",
    ("ICS-INJ-146", "T0826"): "causing a shutdown of operations",
    ("ICS-INJ-146", "T0828"): "causing a shutdown of operations",
    ("ICS-INJ-139", "T0826"): "calls reporting service disruptions",
    ("ICS-INJ-139", "T0832"): "distorted data inputs that impact their ability to monitor",
    ("ICS-INJ-167", "T0822"): "vulnerability in the remote access solution ... used to compromise",
    ("CM-INJ-120", "T1498"): "several million illegitimate web requests per second",
    ("CM-INJ-135", "T1498"): "Customers cannot access the internet because of the DDoS attack",
    ("CM-INJ-147", "T1498"): "used a botnet ... to initiate the attack",
    ("CM-INJ-147", "T1584.005"): "a botnet consisting of approximately 1,000,000 devices",
}
# a threat type -> technique link counts as supported if at least one event has evidence for it
threat_of = events.set_index("inject_id")["threat_type"]
supported = {(threat_of[i], t) for i, t in supported_evidence}
all_links = set(zip(threat_technique_map["threat_type"], threat_technique_map["technique_id"]))
assert supported <= all_links, f"Typo in the supported set: {supported - all_links}"

threat_technique_map["mapping_status"] = [
    "no technique" if pd.isna(tid) else ("supported" if (tt, tid) in supported else "plausible")
    for tt, tid in zip(threat_technique_map["threat_type"], threat_technique_map["technique_id"])]

print(threat_technique_map["mapping_status"].value_counts().to_string())
display(threat_technique_map.dropna(subset=["technique_id"]).head(10))

mapping_status
plausible       35
supported       17
no technique     5


,threat_type,technique_id,domain,justification,mapping_status
0,Suspected Phishing,T1566,enterprise-attack,Emails trick employees into opening attachments or entering credentials.,supported
1,Suspected Phishing,T1598,enterprise-attack,Emails trick employees into opening attachments or entering credentials.,plausible
2,Whaling / Ransomware,T1566,enterprise-attack,Phishing aimed at executives that leads to ransomware.,supported
3,Whaling / Ransomware,T1486,enterprise-attack,Phishing aimed at executives that leads to ransomware.,plausible
4,Ransomware,T1486,enterprise-attack,Files are encrypted and a ransom is demanded; recovery is often blocked.,plausible
5,Ransomware,T1490,enterprise-attack,Files are encrypted and a ransom is demanded; recovery is often blocked.,plausible
6,Suspected Ransomware,T1486,enterprise-attack,"Same behaviour as ransomware, not yet confirmed.",supported
7,Suspected Ransomware,T1490,enterprise-attack,"Same behaviour as ransomware, not yet confirmed.",plausible
8,Extortion Threat,T1657,enterprise-attack,Attackers demand payment; MITRE covers extortion under Financial Theft.,plausible
9,Insider Threat,T1078,enterprise-attack,A trusted employee misuses legitimate accounts and access rights.,supported


#### 5. Validating the mapping
The mapping is checked against both datasets.

*Why:* a typo in a technique ID or a missing threat type would silently drop data in later joins. The checks make sure that (1) every threat type in the CISA data has a mapping entry and every ICS mapping key matches a real ICS threat type, (2) every technique ID exists in MITRE in the right domain, and (3) every mapped technique has at least one mitigation, so each mapped event leads to response options.

In [5]:
# 1. every CISA threat type is covered
missing_types = set(events["threat_type"]) - set(enterprise_map)
assert not missing_types, f"Threat types without a mapping: {missing_types}"

# 1b. every ICS mapping key actually occurs in the ICS manual (catches typos in threat type names)
ics_types = set(events.loc[events["manual_title"] == "Industrial Control Systems", "threat_type"])
unused_ics = set(ics_map) - ics_types
assert not unused_ics, f"ICS mapping keys not found in the ICS events: {unused_ics}"

# 2. every technique exists in the right domain
mapped = threat_technique_map.dropna(subset=["technique_id"])
check = mapped.merge(techniques[["domain", "technique_id", "name", "tactics"]],
                     on=["domain", "technique_id"], how="left")
assert check["name"].notna().all(), f"Unknown IDs: {check.loc[check['name'].isna(), 'technique_id'].tolist()}"

# 3. every technique has mitigations
mit_counts = technique_mitigations.groupby(["domain", "technique_id"]).size()
no_mit = [t for t in mapped.itertuples() if (t.domain, t.technique_id) not in mit_counts.index]
assert not no_mit, f"Techniques without mitigations: {[t.technique_id for t in no_mit]}"

threat_technique_map = threat_technique_map.merge(
    techniques[["domain", "technique_id", "name", "tactics"]].rename(columns={"name": "technique_name"}),
    on=["domain", "technique_id"], how="left")

print("All mapping checks passed.")
print("Threat types:", threat_technique_map["threat_type"].nunique(),
      "| mapped technique links:", len(mapped))

All mapping checks passed.
Threat types: 28 | mapped technique links: 52


#### 6. Linking events to techniques
Each CISA event receives the techniques of its threat type. Events from the Industrial Control Systems manual also receive the ICS techniques.

*Why:* this is the first link of the data model (event → technique). Events that the review in `5_goal1_scenario_analysis` marked as *No Confirmed Incident* (e.g. IT-INJ-93, where an employee is fired but no attack happens) get no techniques. Events without techniques are kept in the event table but listed here, so the gap is visible and can be discussed in the report.

In [6]:
ent_links = events.merge(threat_technique_map[threat_technique_map["domain"] == "enterprise-attack"],
                         on="threat_type")
ics_links = events[events["manual_title"] == "Industrial Control Systems"].merge(
    threat_technique_map[threat_technique_map["domain"] == "ics-attack"], on="threat_type")

event_techniques = (pd.concat([ent_links, ics_links], ignore_index=True)
                    .dropna(subset=["technique_id"])
                    [["inject_id", "manual_title", "threat_type", "threat_category",
                      "domain", "technique_id", "technique_name", "tactics", "mapping_status"]])

# events the review marked as "No Confirmed Incident" describe no attack, so they get no techniques
event_techniques = event_techniques[event_techniques["threat_category"] != "No Confirmed Incident"].copy()

# status per event: supported only where the event text itself describes the technique
pairs = list(zip(event_techniques["inject_id"], event_techniques["technique_id"]))
assert set(supported_evidence) <= set(pairs), f"Evidence for a link that does not exist: {set(supported_evidence) - set(pairs)}"
event_techniques["mapping_status"] = ["supported" if p in supported_evidence else "plausible" for p in pairs]
event_techniques["evidence"] = [supported_evidence.get(p, "") for p in pairs]

# techniques in the "impact" tactic describe consequences rather than how the attack happens
event_techniques["is_impact_technique"] = event_techniques["tactics"].str.contains(r"\bimpact\b", na=False)

mapped_events = event_techniques["inject_id"].nunique()
print(f"Events linked to at least one technique: {mapped_events} / {len(events)}")
print("Event-technique links:", len(event_techniques))
print(event_techniques["mapping_status"].value_counts().to_string())

unmapped = events[~events["inject_id"].isin(event_techniques["inject_id"])]
print("\nEvents without a technique, by threat type:")
print(unmapped["threat_type"].value_counts().to_string())

Events linked to at least one technique: 52 / 66
Event-technique links: 112
mapping_status
plausible    88
supported    24

Events without a technique, by threat type:
threat_type
Security Alert                6
No Specific Threat            4
Device Theft                  1
Accidental Data Disclosure    1
No Specific Cyber Threat      1
Potential Insider Threat      1


#### 7. Linking techniques to mitigations (response options)
The MITRE mitigations of each technique are attached to the events, and the mitigations are ranked per threat category.

*Why:* this completes the chain event → technique → mitigation. The ranking counts how many different events in a threat category a mitigation helps against; mitigations that cover many events are the most useful response options to train in an exercise.

Three MITRE "mitigations" are removed because they are not actions an organization can take in response: *Pre-compromise* (M1056, happens outside the organization before an attack), *Do Not Mitigate* (M1055) and *Mitigation Limited or Not Effective* (M0816).

In [7]:
not_a_response = ["M1056", "M1055", "M0816"]   # Pre-compromise, Do Not Mitigate, Mitigation Limited or Not Effective
usable_mitigations = technique_mitigations[~technique_mitigations["mitigation_id"].isin(not_a_response)]

event_mitigations = event_techniques.merge(
    usable_mitigations, on=["domain", "technique_id"], suffixes=("", "_tm")
)[["inject_id", "threat_category", "domain", "technique_id", "mitigation_id", "mitigation_name"]]

print("Event-mitigation links:", len(event_mitigations))

response_options = (event_mitigations
    .groupby(["threat_category", "mitigation_id", "mitigation_name"])
    .agg(events_covered=("inject_id", "nunique"),
         techniques_covered=("technique_id", "nunique"))
    .reset_index()
    .sort_values(["threat_category", "events_covered", "techniques_covered"],
                 ascending=[True, False, False]))

# top 3 response options per threat category
display(response_options.groupby("threat_category").head(3).reset_index(drop=True))

Event-mitigation links: 423


,threat_category,mitigation_id,mitigation_name,events_covered,techniques_covered
0,Data Exposure,M1037,Filter Network Traffic,7,3
1,Data Exposure,M1018,User Account Management,7,2
2,Data Exposure,M1022,Restrict File and Directory Permissions,7,2
3,Denial of Service,M1037,Filter Network Traffic,4,3
4,Insider Threat / Unauthorized Access,M1032,Multi-factor Authentication,8,5
5,Insider Threat / Unauthorized Access,M1018,User Account Management,6,3
6,Insider Threat / Unauthorized Access,M1026,Privileged Account Management,6,3
7,Other Security Incident,M1022,Restrict File and Directory Permissions,1,1
8,Other Security Incident,M1029,Remote Data Storage,1,1
9,Other Security Incident,M1031,Network Intrusion Prevention,1,1


#### 8. Comparing the team-created mapping with TF-IDF
For each mapped event, it is checked whether any of the team-mapped techniques appears among the TF-IDF top 5 matches.

*Why:* this gives a number for how unreliable the TF-IDF approach is, which can be reported to justify the choice of a manual mapping. Only Enterprise techniques are compared, because TF-IDF was run on Enterprise only.

In [8]:
top5 = similarities.argsort(axis=1)[:, ::-1][:, :5]
tfidf_top5 = {events.loc[i, "inject_id"]: set(ent.loc[top5[i], "technique_id"]) for i in range(len(events))}

expert = (event_techniques[event_techniques["domain"] == "enterprise-attack"]
          .groupby("inject_id")["technique_id"].apply(set))

hits = [bool(expert[e] & tfidf_top5[e]) for e in expert.index]
print(f"Events where TF-IDF top 5 contains a team-mapped technique: {sum(hits)} / {len(hits)} "
      f"({sum(hits) / len(hits):.0%})")

Events where TF-IDF top 5 contains a team-mapped technique: 13 / 52 (25%)


#### 9. Building the scenario model table
One readable row per event: the threat, the MITRE techniques, the top response options and the business impact.

*Why:* this is the combined view of the data model that Goal 2 asks for. Plausible technique links are marked *(plausible)*, so they are not mistaken for confirmed ones. `impact_status` shows whether the consequence is described as occurring or only potential. The response options are possible MITRE mitigations for the techniques, not responses the manuals prove to work.

Enterprise and ICS ATT&CK have some mitigations with the same name (e.g. *Data Backup*), so names are de-duplicated before taking the top 5.

In [9]:
techs_per_event = (event_techniques
                   .assign(label=event_techniques["technique_id"] + " " + event_techniques["technique_name"]
                           + event_techniques["mapping_status"].map({"plausible": " (plausible)"}).fillna(""))
                   .groupby("inject_id")["label"].apply("; ".join)
                   .rename("mitre_techniques"))

top_mits = (event_mitigations.groupby(["inject_id", "mitigation_id", "mitigation_name"]).size()
            .reset_index(name="n").sort_values(["inject_id", "n"], ascending=[True, False])
            .drop_duplicates(["inject_id", "mitigation_name"])   # Enterprise and ICS can share a mitigation name
            .groupby("inject_id").head(5)
            .groupby("inject_id")["mitigation_name"].apply("; ".join)
            .rename("response_options"))

scenario_model = (events[["inject_id", "manual_title", "section", "day", "threat_type",
                          "threat_category", "business_impact", "impact_category", "impact_status"]]
                  .merge(techs_per_event, on="inject_id", how="left")
                  .merge(top_mits, on="inject_id", how="left"))

display(scenario_model[scenario_model["mitre_techniques"].notna()].head(8))

,inject_id,manual_title,section,day,threat_type,threat_category,business_impact,impact_category,impact_status,mitre_techniques,response_options
1,HC-INJ-89,Healthcare,Module 1,Day 9,Suspected Phishing,Phishing,Potential credential theft and malware infection through suspicious emails and attachments,Security Risk,Potential or unconfirmed,T1566 Phishing; T1598 Phishing for Information (plausible),User Training; Software Configuration; Restrict Web-Based Content; Network Intrusion Prevention; Audit
3,HC-INJ-119,Healthcare,Module 1,Day 13,Vulnerability Exposure,Vulnerability,Potential compromise of clinical devices and systems if the vulnerability remains unpatched,Security Risk,Potential or unconfirmed,T1190 Exploit Public-Facing Application (plausible); T1068 Exploitation for Privilege Escalation (plausible),Application Isolation and Sandboxing; Exploit Protection; Update Software; Vulnerability Scanning; Threat Intelligen...
4,HC-INJ-126,Healthcare,Module 1,Day 15,System Disruption (Unconfirmed Cause),System Disruption,MRI system delays causing patient appointments to run more than two hours behind schedule,Operational Disruption,Described as occurring,T1489 Service Stop (plausible); T1499 Endpoint Denial of Service (plausible),User Account Management; Restrict File and Directory Permissions; Restrict Registry Permissions; Network Segmentatio...
5,HC-INJ-135,Healthcare,Module 2,Day 20,Unauthorized Access Risk,Insider Threat / Unauthorized Access,Potential misuse of former employees' active IT accounts; accounts were deactivated,Security Risk,Potential or unconfirmed,T1078 Valid Accounts (plausible); T1621 Multi-Factor Authentication Request Generation (plausible),User Training; Multi-factor Authentication; Account Use Policies; Application Developer Guidance; Active Directory C...
6,HC-INJ-137,Healthcare,Module 2,Day 23 - Morning,Data Integrity Issue,Other Security Incident,Incorrect patient records creating risks to patient safety and clinical decisions,Patient Safety,Potential or unconfirmed,T1565.001 Stored Data Manipulation (plausible),Restrict File and Directory Permissions; Remote Data Storage; Encrypt Sensitive Information
7,HC-INJ-139,Healthcare,Module 2,Day 23 - Afternoon,Medical Device Malfunction,System Disruption,Inaccurate monitoring data and incorrect infusion rates threatening patient safety,Patient Safety,Potential or unconfirmed,T1565 Data Manipulation (plausible),Restrict File and Directory Permissions; Remote Data Storage; Network Segmentation; Encrypt Sensitive Information
8,HC-INJ-156,Healthcare,Module 2,Day 25,Ransomware,Ransomware,"Files unavailable, $1 million ransom demand, and threatened exposure of patient data",Operational Disruption,Described as occurring,T1486 Data Encrypted for Impact (plausible); T1490 Inhibit System Recovery (plausible),Data Backup; User Account Management; Operating System Configuration; Execution Prevention; Behavior Prevention on E...
9,HC-INJ-159,Healthcare,Module 2,Day 26,Data Breach,Data Exposure,"Patient information exposed; extortion attempts, complaints, and threatened legal action",Data Exposure,Described as occurring,T1530 Data from Cloud Storage (plausible); T1567 Exfiltration Over Web Service (plausible),User Account Management; Restrict Web-Based Content; Restrict File and Directory Permissions; Multi-factor Authentic...


#### 10. Saving the results
The mapping and the linked tables are saved.

*Why:* each table is one part of the data model, so it can be reused separately:
- `threat_technique_map.csv`: the team-created mapping with justifications and mapping status (for the report)
- `cisa_event_techniques.csv`: event → technique
- `cisa_event_mitigations.csv`: event → technique → mitigation
- `response_options.csv`: ranked mitigations per threat category
- `scenario_model.csv`: the combined scenario → response → outcome view

In [10]:
outputs = {
    "threat_technique_map.csv": threat_technique_map,
    "cisa_event_techniques.csv": event_techniques,
    "cisa_event_mitigations.csv": event_mitigations,
    "response_options.csv": response_options,
    "scenario_model.csv": scenario_model,
}
for name, df in outputs.items():
    df.to_csv(data_dir / name, index=False, encoding="utf-8-sig")
    print(f"{name}: {len(df)} rows")

threat_technique_map.csv: 57 rows
cisa_event_techniques.csv: 112 rows
cisa_event_mitigations.csv: 423 rows
response_options.csv: 124 rows
scenario_model.csv: 66 rows


#### 11. Goal 1 summary table
One row per threat category: how many events, the most common impacts, and the top 3 MITRE responses.

*Why:* a compact table for the poster that combines both datasets: CISA gives the threats and consequences, MITRE gives the possible responses. Events without a confirmed incident are left out.

In [11]:
# Only real incidents (leave out alerts and "nothing happened" events)
incidents = events[events["threat_category"] != "No Confirmed Incident"]

summary = incidents.groupby("threat_category").agg(
    events=("inject_id", "count"),
    main_impacts=("impact_category", lambda s: "; ".join(s.value_counts().index[:2])),
)

# response_options is already sorted best-first, so take the top 3 per category
top_responses = (response_options
                 .drop_duplicates(["threat_category", "mitigation_name"])
                 .groupby("threat_category").head(3)
                 .groupby("threat_category")["mitigation_name"].apply("; ".join)
                 .rename("top_responses"))

goal1_summary = summary.join(top_responses).sort_values("events", ascending=False)
goal1_summary.to_csv(data_dir / "goal1_summary.csv", encoding="utf-8-sig")
goal1_summary

,events,main_impacts,top_responses
threat_category,,,
Ransomware,10,Operational Disruption; Reputational Damage,User Account Management; Data Backup; Behavior Prevention on Endpoint
Data Exposure,8,Data Exposure; Reputational Damage,Filter Network Traffic; User Account Management; Restrict File and Directory Permissions
Insider Threat / Unauthorized Access,8,Security Risk; Data Loss,Multi-factor Authentication; User Account Management; Privileged Account Management
System Disruption,6,Operational Disruption; Patient Safety,Restrict File and Directory Permissions; Network Segmentation; User Account Management
Phishing,5,Security Risk,User Training; Software Configuration; Restrict Web-Based Content
Third-Party / Supply Chain,5,Security Risk; No Direct Impact,User Account Management; Multi-factor Authentication; Network Segmentation
Vulnerability,5,Security Risk; Financial Cost,Application Isolation and Sandboxing; Exploit Protection; Update Software
Denial of Service,4,Operational Disruption; Reputational Damage,Filter Network Traffic
Other Security Incident,3,Security Risk; Patient Safety,Restrict File and Directory Permissions; Remote Data Storage; Network Intrusion Prevention


#### 12. Goal 2 table per exercise
Each event is linked to its exercise (EX-01 … EX-16), and the chain threat → techniques → responses → outcomes is summarised per exercise.

*Why:* an exercise is what a trainer actually runs, so this is the scenario level of the data model. `scenario_model.csv` is saved again with the new `exercise_id` column.

`outcomes` marks consequences that are only potential or unconfirmed with *(potential)*, using `impact_status` from the Goal 1 review.

Note: `cisa_exercises.csv` (notebook 4) still has the original, unreviewed threat and impact labels, because notebook 4 runs before the review in notebook 5. `goal2_exercise_model.csv` is the exercise table with the reviewed labels.

In [12]:
exercises = pd.read_csv(data_dir / "cisa_exercises.csv")

# Which exercise each event belongs to (event_ids is a "; "-separated list)
event_exercise = (exercises[["exercise_id", "event_ids"]]
                  .assign(inject_id=exercises["event_ids"].str.split("; "))
                  .explode("inject_id")[["exercise_id", "inject_id"]])

# Add exercise_id to the event-level scenario model
scenario_model = (scenario_model.drop(columns="exercise_id", errors="ignore")
                  .merge(event_exercise, on="inject_id", how="left"))
assert scenario_model["exercise_id"].notna().all(), "Some events are not in any exercise"

def join_unique(values):
    """Join "; "-lists into one list without repeats. A "(plausible)" or "(potential)" copy is dropped
    if the same item also appears without it (supported technique / outcome described as occurring)."""
    items = list(dict.fromkeys(x for v in values.dropna() for x in str(v).split("; ")))
    weaker = (" (plausible)", " (potential)")
    return "; ".join(x for x in items
                     if not any(x.endswith(w) and x.removesuffix(w) in items for w in weaker))

# outcome with its certainty, so potential consequences are not shown as if they happened
scenario_model["outcome"] = scenario_model["impact_category"] + scenario_model["impact_status"].map(
    {"Potential or unconfirmed": " (potential)"}).fillna("")

# Top 5 responses per exercise: mitigations that help against the most events in it
exercise_responses = (event_mitigations
    .merge(event_exercise, on="inject_id")
    .groupby(["exercise_id", "mitigation_name"])["inject_id"].nunique()
    .reset_index(name="events_covered")
    .sort_values(["exercise_id", "events_covered"], ascending=[True, False])
    .groupby("exercise_id").head(5)
    .groupby("exercise_id")["mitigation_name"].apply("; ".join)
    .rename("top_responses"))

goal2_exercise_model = (scenario_model
    .groupby("exercise_id")
    .agg(manual_title=("manual_title", "first"),
         section=("section", "first"),
         events=("inject_id", "count"),
         threat_types=("threat_type", join_unique),
         mitre_techniques=("mitre_techniques", join_unique),
         outcomes=("outcome", join_unique))
    .join(exercise_responses)
    .reset_index())

scenario_model.to_csv(data_dir / "scenario_model.csv", index=False, encoding="utf-8-sig")
goal2_exercise_model.to_csv(data_dir / "goal2_exercise_model.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", None)
goal2_exercise_model

,exercise_id,manual_title,section,events,threat_types,mitre_techniques,outcomes,top_responses
0,EX-01,Healthcare,Module 1,5,Security Alert; Suspected Phishing; No Specific Threat; Vulnerability Exposure; System Disruption (Unconfirmed Cause),T1566 Phishing; T1598 Phishing for Information (plausible); T1190 Exploit Public-Facing Application (plausible); T1068 Exploitation for Privilege Escalation (plausible); T1489 Service Stop (plausible); T1499 Endpoint Denial of Service (plausible),No Direct Impact; Security Risk (potential); Operational Disruption,Filter Network Traffic; Network Segmentation; Antivirus/Antimalware; Application Isolation and Sandboxing; Audit
1,EX-02,Healthcare,Module 2,6,Unauthorized Access Risk; Data Integrity Issue; Medical Device Malfunction; Ransomware; Data Breach,T1078 Valid Accounts (plausible); T1621 Multi-Factor Authentication Request Generation (plausible); T1565.001 Stored Data Manipulation (plausible); T1565 Data Manipulation (plausible); T1486 Data Encrypted for Impact (plausible); T1490 Inhibit System Recovery (plausible); T1530 Data from Cloud Storage (plausible); T1567 Exfiltration Over Web Service (plausible),Security Risk (potential); Patient Safety (potential); Operational Disruption; Data Exposure; Reputational Damage,Encrypt Sensitive Information; Restrict File and Directory Permissions; User Account Management; Multi-factor Authentication; Audit
2,EX-03,Vendor,Module 1,4,Supply Chain Compromise; No Specific Threat; Third-Party Data Breach,T1195.002 Compromise Software Supply Chain (plausible); T1199 Trusted Relationship,No Direct Impact; Security Risk (potential),Multi-factor Authentication; Network Segmentation; User Account Management; Update Software; Vulnerability Scanning
3,EX-04,Vendor,Module 2 – 6 months later,4,System Disruption (Unconfirmed Cause); Supply Chain Compromise; Suspicious Network Activity,T1489 Service Stop (plausible); T1499 Endpoint Denial of Service (plausible); T1195.002 Compromise Software Supply Chain; T1199 Trusted Relationship; T1041 Exfiltration Over C2 Channel (plausible),Operational Disruption; Security Risk (potential),Network Segmentation; User Account Management; Filter Network Traffic; Out-of-Band Communications Channel; Restrict File and Directory Permissions
4,EX-05,Ransomware,Module 1,4,Security Alert; Vulnerability Exposure; Device Theft; Suspected Phishing,T1190 Exploit Public-Facing Application (plausible); T1068 Exploitation for Privilege Escalation (plausible); T1566 Phishing; T1598 Phishing for Information (plausible),No Direct Impact; Security Risk (potential),Antivirus/Antimalware; Application Isolation and Sandboxing; Audit; Execution Prevention; Exploit Protection
5,EX-06,Ransomware,Module 2,4,Suspected Data Exfiltration; Ransomware; Data Breach,T1048 Exfiltration Over Alternative Protocol; T1071.004 DNS (plausible); T1486 Data Encrypted for Impact (plausible); T1490 Inhibit System Recovery (plausible); T1530 Data from Cloud Storage (plausible); T1567 Exfiltration Over Web Service (plausible),Operational Disruption; Data Exposure; Reputational Damage (potential),User Account Management; Behavior Prevention on Endpoint; Data Backup; Data Loss Prevention; Execution Prevention
6,EX-07,Insider Threat,Module 1,5,Vulnerability Exposure; Accidental Data Disclosure; No Specific Cyber Threat; Potential Insider Threat,T1190 Exploit Public-Facing Application (plausible); T1068 Exploitation for Privilege Escalation (plausible),Security Risk (potential); Financial Cost (potential); Data Exposure; Personnel Consequences,Application Isolation and Sandboxing; Execution Prevention; Exploit Protection; Filter Network Traffic; Limit Access to Resource Over Network
7,EX-08,Insider Threat,Module 2,7,Vulnerability Exposure; Potential Insider Threat; Unauthorized File Modification; Unauthorized Access; Suspicious Third-Party Activity; Unauthorized File Deletion,T1190 Exploit Public-Facing Application (plausible); T1078 Valid Accounts (plausible); T1565.001 Sto